[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 Hard: INT8 Quantized Linear

Реализуйте **post-training quantized linear layer** с INT8-весами. Идея: после обучения заменить каждый floating-point вес небольшим целым кодом и отдельным scale, а при forward восстановить приближённое значение.

## Симметричная per-output-channel квантизация

Матрица `weight` линейного слоя имеет форму `(out_features, in_features)`. Для каждой строки выбирается свой масштаб $s_o=\max_i|w_{o,i}|/127$. Затем $q_{o,i}=\operatorname{round}(w_{o,i}/s_o)$ ограничивается диапазоном INT8, а приближение равно $\hat w_{o,i}=q_{o,i}s_o$. Отдельный scale на output channel лучше использует диапазон, если строки имеют разные амплитуды.

Например, для строки `[-1.0, 0.5]` scale примерно `1/127`; коды близки к `[-127, 64]`, а после dequantization второй вес станет около `0.504`. Ошибка округления ожидаема, поэтому результат сравнивается с float-слоем с допуском, а не побитово.

Хотя dtype int8 допускает `-128`, симметричная схема использует делитель 127, чтобы положительная и отрицательная амплитуда были симметричны. Строка из одних нулей требует безопасного ненулевого scale, иначе деление даст NaN.

### Signature
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul

> Эта учебная версия деквантует веса перед обычным floating-point matmul. Она демонстрирует формат хранения и quantization error, но сама по себе не ускоряет вычисления как специализированное INT8-ядро.

## План реализации

1. Вычислите column-shaped scale `(out_features,1)` с защитой zero rows.
2. Округлите, ограничьте и преобразуйте веса в `torch.int8`.
3. Зарегистрируйте quantized weight и scale как buffers; bias сохраните в соответствии с контрактом слоя.
4. В forward деквантуйте с dtype входа, выполните linear operation и добавьте bias.

## Быстрые проверки

- `weight_int8.dtype == torch.int8`, и он отсутствует среди trainable parameters.
- Scale имеет по одному значению на строку и корректно broadcast по `in_features`.
- Zero weight matrix даёт конечный output, равный bias при его наличии.
- Ответ близок к исходному `F.linear(x, weight, bias)`.

## Частые ошибки

- Взять максимум по `dim=0` и получить per-input-channel схему.
- Потерять singleton-ось scale, из-за чего broadcasting работает не по строкам.
- Сделать INT8 buffer параметром с градиентом.
- Ожидать точного равенства float и quantized outputs.

## Материалы для глубокого изучения

- [PyTorch Quantization recipe](https://docs.pytorch.org/tutorials/recipes/quantization.html) — базовые схемы и практический workflow.
- [PyTorch quantization support](https://docs.pytorch.org/docs/stable/quantization-support) — API, observers и mapping.
- [torchao: writing a quantized tensor subclass](https://docs.pytorch.org/ao/stable/eager_tutorials/subclass_basic.html) — устройство современных quantized tensor abstractions.

## Вопросы для самопроверки

1. Почему scale формы `(out,1)`, а не `(in,)`?
2. Откуда возникает quantization error?
3. Почему эта реализация экономит хранение весов, но не гарантирует ускорение matmul?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        pass  # quantize weight, register buffers

    def forward(self, x):
        pass  # dequantize and matmul

In [ ]:
# 🧪 Debug
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('Output:', q(x).shape)
print('dtype:', q.weight_int8.dtype)
print('Max quant error:', (w - q.weight_int8.float() * q.scale).abs().max().item())

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('int8_quantization')